<a href="https://colab.research.google.com/github/Harshitha-hunny/part-4/blob/main/part4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [31]:
# Install FastAPI and Uvicorn
!pip install fastapi uvicorn pydantic pandas scikit-learn joblib httpx -q

In [32]:
%%writefile requirements.txt
fastapi
uvicorn
pydantic
pandas
scikit-learn
joblib
httpx
nest_asyncio

Overwriting requirements.txt


In [33]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import joblib

# Using the feature_columns list defined previously in the Pydantic models
feature_columns = [
    'recency_days', 'frequency_180d', 'monetary_180d', 'return_rate_180d',
    'avg_discount_pct_180d', 'avg_rating_180d', 'category_diversity_180d',
    'ticket_count_90d', 'negative_ticket_rate_90d', 'avg_resolution_hours_90d',
    'days_since_signup', 'sessions_30d', 'product_views_30d', 'cart_adds_30d',
    'wishlist_adds_30d', 'abandoned_carts_30d', 'email_opens_30d', 'campaign_clicks_30d',
    'last_visit_days_ago', 'city_tier_Tier 2', 'city_tier_Tier 3',
    'age_group_25-34', 'age_group_35-44', 'age_group_45+', 'acquisition_channel_Direct',
    'acquisition_channel_Google Search', 'acquisition_channel_Influencer', 'acquisition_channel_Organic',
    'acquisition_channel_Referral', 'acquisition_channel_Social Media', 'loyalty_tier_Platinum',
    'loyalty_tier_Silver', 'preferred_category_Fragrance', 'preferred_category_Hair Care',
    'preferred_category_Makeup', 'preferred_category_Skin Care', 'preferred_category_Wellness',
    'marketing_consent_Yes'
]

# Generate dummy data for X (features) and y (target)
np.random.seed(42)
dummy_data = {
    col: np.random.rand(1000) * 10 if 'days' in col or 'rate' in col or 'hours' in col
    else np.random.randint(0, 2) if 'tier' in col or 'consent' in col or 'group' in col or 'channel' in col or 'tier' in col or 'category' in col
    else np.random.rand(1000) * 100
    for col in feature_columns
}
df_dummy = pd.DataFrame(dummy_data)

# Ensure one-hot encoded columns are 0 or 1
for col in feature_columns:
    if 'city_tier' in col or 'age_group' in col or 'acquisition_channel' in col or 'loyalty_tier' in col or 'preferred_category' in col or 'marketing_consent' in col:
        df_dummy[col] = np.random.randint(0, 2, 1000)

# Add a target variable 'churn' (binary classification)
df_dummy['churn'] = np.random.randint(0, 2, 1000)

# Split into training and testing sets
X = df_dummy[feature_columns]
y = df_dummy['churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Generated X_train shape: {X_train.shape}")
print(f"Generated y_train shape: {y_train.shape}")
print(f"Generated X_test shape: {X_test.shape}")
print(f"Generated y_test shape: {y_test.shape}")

# Train a simple Logistic Regression model
model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train, y_train)

# Save the trained model
joblib.dump(model, 'model.pkl')

print("Dummy data generated, Logistic Regression model trained, and saved as 'model.pkl'.")

Generated X_train shape: (800, 38)
Generated y_train shape: (800,)
Generated X_test shape: (200, 38)
Generated y_test shape: (200,)
Dummy data generated, Logistic Regression model trained, and saved as 'model.pkl'.


In [34]:
from pydantic import BaseModel, create_model
from typing import List, Dict, Any

field_definitions = {col: (float, ...) for col in feature_columns}

ChurnPredictionInput = create_model('ChurnPredictionInput', **field_definitions)

class ChurnPredictionOutput(BaseModel):
    churn_probability: float
    predicted_class: int
    risk_explanation: str

class BatchChurnPredictionInput(BaseModel):
    instances: List[ChurnPredictionInput]

class BatchChurnPredictionOutput(BaseModel):
    predictions: List[ChurnPredictionOutput]

print("Pydantic models for churn prediction input and output defined.")

Pydantic models for churn prediction input and output defined.


In [35]:
%%writefile app.py

import joblib
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, create_model # Import create_model
from typing import List, Dict, Any


feature_columns = [
    'recency_days', 'frequency_180d', 'monetary_180d', 'return_rate_180d',
    'avg_discount_pct_180d', 'avg_rating_180d', 'category_diversity_180d',
    'ticket_count_90d', 'negative_ticket_rate_90d', 'avg_resolution_hours_90d',
    'days_since_signup', 'sessions_30d', 'product_views_30d', 'cart_adds_30d',
    'wishlist_adds_30d', 'abandoned_carts_30d', 'email_opens_30d', 'campaign_clicks_30d',
    'last_visit_days_ago', 'city_tier_Tier 2', 'city_tier_Tier 3',
    'age_group_25-34', 'age_group_35-44', 'age_group_45+', 'acquisition_channel_Direct',
    'acquisition_channel_Google Search', 'acquisition_channel_Influencer', 'acquisition_channel_Organic',
    'acquisition_channel_Referral', 'acquisition_channel_Social Media', 'loyalty_tier_Platinum',
    'loyalty_tier_Silver', 'preferred_category_Fragrance', 'preferred_category_Hair Care',
    'preferred_category_Makeup', 'preferred_category_Skin Care', 'preferred_category_Wellness',
    'marketing_consent_Yes'
]

# Create a dictionary for field definitions, setting all as floats for now
# Assuming all numerical features and one-hot encoded categoricals can be represented as floats
field_definitions = {col: (float, ...) for col in feature_columns}
ChurnPredictionInput = create_model('ChurnPredictionInput', **field_definitions) # Use create_model here
# --- END Manual feature list ---


class ChurnPredictionOutput(BaseModel):
    churn_probability: float
    predicted_class: int
    risk_explanation: str

class BatchChurnPredictionInput(BaseModel):
    instances: List[ChurnPredictionInput]

class BatchChurnPredictionOutput(BaseModel):
    predictions: List[ChurnPredictionOutput]

app = FastAPI()

# Load the trained model
try:
    model = joblib.load('model.pkl')
    print("Model loaded successfully.")
except Exception as e:
    raise RuntimeError(f"Could not load model: {e}")

@app.get("/health")
async def health():
    """Health check endpoint."""
    return {"status": "ok", "message": "Churn prediction service is up and running!"}

@app.post("/predict", response_model=ChurnPredictionOutput)
async def predict(input_data: ChurnPredictionInput):
    """Predict churn for a single customer instance."""
    df = pd.DataFrame([input_data.dict()])

    # Ensure columns are in the same order as during training
    df = df[feature_columns]

    churn_probability = model.predict_proba(df)[:, 1][0]
    predicted_class = int(model.predict(df)[0])

    if predicted_class == 1:
        risk_explanation = "High risk of churn. Consider proactive retention strategies."
    else:
        risk_explanation = "Low risk of churn. Continue monitoring customer engagement."

    return ChurnPredictionOutput(
        churn_probability=churn_probability,
        predicted_class=predicted_class,
        risk_explanation=risk_explanation
    )

@app.post("/batch_predict", response_model=BatchChurnPredictionOutput)
async def batch_predict(batch_input: BatchChurnPredictionInput):
    """Predict churn for multiple customer instances."""
    predictions_output = []
    for input_data in batch_input.instances:
        df = pd.DataFrame([input_data.dict()])

        # Ensure columns are in the same order as during training
        df = df[feature_columns]

        churn_probability = model.predict_proba(df)[:, 1][0]
        predicted_class = int(model.predict(df)[0])

        if predicted_class == 1:
            risk_explanation = "High risk of churn. Consider proactive retention strategies."
        else:
            risk_explanation = "Low risk of churn. Continue monitoring customer engagement."

        predictions_output.append(ChurnPredictionOutput(
            churn_probability=churn_probability,
            predicted_class=predicted_class,
            risk_explanation=risk_explanation
        ))

    return BatchChurnPredictionOutput(predictions=predictions_output)

Overwriting app.py


In [36]:
# To run FastAPI in Colab, we need to set up a tunnel or use a tool like `ngrok`.
# For demonstration within Colab, we'll run uvicorn in the background
# and use `nest_asyncio` for compatibility with Colab's event loop.

import uvicorn
import nest_asyncio
import threading
import time

# Import the 'app' object from the app.py file
# This assumes app.py has been successfully written and contains 'app = FastAPI()'
try:
    from app import app
    print("FastAPI 'app' object imported successfully from app.py")
except ImportError as e:
    print(f"Error importing app from app.py: {e}. Ensure app.py was written correctly and contains `app = FastAPI()`.")
    raise # Re-raise the error if import fails critical for next steps
except Exception as e:
    print(f"An unexpected error occurred during import: {e}")
    raise

nest_asyncio.apply()

def run_fastapi_thread(app_instance):
    """Function to run FastAPI app, passed as an argument to ensure scope."""
    uvicorn.run(app_instance, host="0.0.0.0", port=8000)

# Start the FastAPI server in a new thread
# We pass the imported 'app' object to the thread's target function
api_thread = threading.Thread(target=run_fastapi_thread, args=(app,))
api_thread.daemon = True # Allow main program to exit even if thread is still running
api_thread.start()

print("FastAPI app is attempting to run in the background on http://0.0.0.0:8000")
print("You might need to expose this port using a tool like `ngrok` for external access.")

# Add a small delay to allow the server to fully start before tests try to connect
time.sleep(2) # Added a small delay for server startup

FastAPI 'app' object imported successfully from app.py
FastAPI app is attempting to run in the background on http://0.0.0.0:8000
You might need to expose this port using a tool like `ngrok` for external access.


INFO:     Started server process [8433]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
